# Part 1, extension 3a check: was the chosen window really the busiest?

When notebook 07 searched for the 0.25° window with the most forest loss, Python warned "invalid value encountered in subtract". The warning means some cells of the 1 km search grid had no valid number. A window containing such a cell gets no score, so it could never be picked, even if it had more loss than the window that was chosen.

This notebook rereads the same 1 km search grid (one small Earth Engine request, no images) and checks three things:

1. **Where the missing cells are**, and whether any are inside Peru.
2. **The best window among windows with no missing cells**, which should be the one notebook 07 used.
3. **Whether any window with missing cells could beat it.** Every missing cell is counted as 100% lost, the most it could possibly be. If no window wins even then, the choice is confirmed.

It changes nothing. If the window turns out not to be the busiest, it says so, and deciding what to do about it comes next.

In [ ]:
EE_PROJECT = "earth-01-final"
SEARCH_BOX = [-71.5, -13.5, -68.7, -11.0]   # as in notebook 07
WINDOW_DEG = 0.25
COARSE_DEG = 0.01
TREECOVER_MIN = 30
GRID_JSON = "../data/cleaned/part1_map_grid.json"

import json
import numpy as np
import ee

ee.Authenticate()
ee.Initialize(project=EE_PROJECT, opt_url="https://earthengine-highvolume.googleapis.com")
ee.data.setDeadline(180_000)

USED = json.load(open(GRID_JSON))["window"]      # the window notebook 07 actually downloaded
print("Window used by notebooks 07 and 08:", [float(v) for v in USED])

In [ ]:
# Same search grid as notebook 07, section 3 (code copied unchanged)
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
treecover = hansen.select("treecover2000").unmask(0)
lossyear = hansen.select("lossyear").unmask(0)
datamask = hansen.select("datamask").unmask(0)
forest2000 = treecover.gte(TREECOVER_MIN).And(datamask.eq(1))
peru = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Peru")).geometry()
recent = forest2000.And(lossyear.gte(19)).And(lossyear.lte(23))
search = (recent.toFloat().reduceResolution(ee.Reducer.mean(), maxPixels=4096).rename("recent")
          .addBands(ee.Image.constant(1).clip(peru).unmask(0).toFloat().rename("peru")))

w, s, e, n = SEARCH_BOX
W, H = int(round((e - w) / COARSE_DEG)), int(round((n - s) / COARSE_DEG))
arr = ee.data.computePixels({
    "expression": search, "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": W, "height": H},
             "affineTransform": {"scaleX": COARSE_DEG, "shearX": 0, "translateX": w,
                                 "shearY": 0, "scaleY": -COARSE_DEG, "translateY": n},
             "crsCode": "EPSG:4326"}})
recent_map, peru_map = arr["recent"].astype(float), arr["peru"].astype(float)

## 1. Where are the missing cells?

In [ ]:
bad = ~np.isfinite(recent_map) | ~np.isfinite(peru_map)
print(f"Missing cells: {bad.sum()} of {bad.size} ({bad.mean():.2%})")
if bad.any():
    rows, cols = np.where(bad)
    print(f"  They span lon {w + cols.min() * COARSE_DEG:.2f} to {w + (cols.max() + 1) * COARSE_DEG:.2f}, "
          f"lat {n - (rows.max() + 1) * COARSE_DEG:.2f} to {n - rows.min() * COARSE_DEG:.2f}")
    in_peru = bad & (np.nan_to_num(peru_map, nan=0) >= 0.5)
    print(f"  Missing cells marked as inside Peru: {in_peru.sum()}")

## 2 and 3. Best window, with missing cells left out and with missing cells counted as fully lost

In [ ]:
def window_sums(a, k):     # as in notebook 07
    c = np.pad(a, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    return c[k:, k:] - c[:-k, k:] - c[k:, :-k] + c[:-k, :-k]

k = int(round(WINDOW_DEG / COARSE_DEG))
peru_ok = np.nan_to_num(peru_map, nan=0.0)
inside = window_sums(peru_ok, k) >= k**2 - 0.5                 # fully inside Peru
has_bad = window_sums(bad.astype(float), k) > 0.5              # contains a missing cell

share_clean = window_sums(np.where(bad, 0.0, recent_map), k) / k**2
share_upper = window_sums(np.where(bad, 1.0, recent_map), k) / k**2   # missing cells counted as 100% lost

def to_window(i, j):
    return [float(round(w + j * COARSE_DEG, 4)), float(round(n - (i + k) * COARSE_DEG, 4)),
            float(round(w + (j + k) * COARSE_DEG, 4)), float(round(n - i * COARSE_DEG, 4))]

clean = inside & ~has_bad
assert clean.any(), "no window inside Peru is free of missing cells"
i, j = np.unravel_index(np.argmax(np.where(clean, share_clean, -1)), share_clean.shape)
best_clean = to_window(i, j)
print(f"Busiest window with no missing cells: {best_clean}, loss {share_clean[i, j]:.2%}")

i_used = int(round((n - USED[3]) / COARSE_DEG)); j_used = int(round((USED[0] - w) / COARSE_DEG))
used_share = share_clean[i_used, j_used]
print(f"Window used by notebooks 07 and 08:   {[float(v) for v in USED]}, loss {used_share:.2%}"
      f"{'  (contains missing cells)' if has_bad[i_used, j_used] else ''}")

rivals = inside & has_bad & (share_upper > used_share)
print(f"Windows with missing cells that could beat it, even counting every missing cell as fully lost: {rivals.sum()}")

same = np.allclose(best_clean, USED, atol=1e-6)
if same and not rivals.any():
    print("\nConfirmed: the window used is the busiest 0.25° window inside Peru in the search rectangle.")
elif same:
    print("\nNot fully confirmed: the window used is the busiest of the clean windows, but some windows with "
          "missing cells could be busier. Send this output to Claude before changing anything.")
else:
    print("\nNot confirmed: a different window has more loss. Send this output to Claude before changing anything.")